# HAVING + Window Functions

**SOLUTIONS** — HAVING + Window Functions

## Critical fact (SQLite and most SQL engines)

> **You cannot put a window function inside `HAVING` (or `WHERE`).**

```sql
-- ❌ ILLEGAL
SELECT Country, COUNT(*) AS cnt,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
FROM customers
GROUP BY Country
HAVING RANK() OVER (ORDER BY COUNT(*) DESC) <= 3;   -- error: misuse of window function
```

## Why?

Logical processing order of a SELECT statement:

```
1. FROM / JOIN
2. WHERE
3. GROUP BY
4. HAVING          ← aggregates are available here
5. WINDOW          ← window functions are calculated here
6. SELECT
7. DISTINCT
8. ORDER BY
9. LIMIT / OFFSET
```

Window functions run **after** HAVING. Therefore HAVING cannot “see” them.

## Three practical patterns you will learn

| Pattern | When to use |
|---------|-------------|
| **A. HAVING then Window** | Filter groups first, then rank / number the surviving groups |
| **B. Window inside a subquery/CTE, then filter** | Filter on a window result (the SQLite equivalent of `QUALIFY`) |
| **C. Aggregate + Window in same SELECT** | Show both group aggregates and window calculations side-by-side |

All exercises use the Chinook database.


## Exercise 1 – Illegal attempt (observe the error)

**Goal:** See the error message SQLite gives when a window function is placed in HAVING.

### Instructions

Run the following (intentionally wrong) query and read the error:

```sql
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS CountryRank
FROM customers
GROUP BY Country
HAVING RANK() OVER (ORDER BY COUNT(*) DESC) <= 3;
```

You should receive something like: `misuse of window function RANK()`.


In [ ]:
-- This query is INTENTIONALLY INVALID.
-- Run it to see the error: "misuse of window function RANK()"
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS CountryRank
FROM customers
GROUP BY Country
HAVING RANK() OVER (ORDER BY COUNT(*) DESC) <= 3;


**Solution**

```sql
-- This query is INTENTIONALLY INVALID.
-- Run it to see the error: "misuse of window function RANK()"
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS CountryRank
FROM customers
GROUP BY Country
HAVING RANK() OVER (ORDER BY COUNT(*) DESC) <= 3;
```

> This query is meant to fail. Running it demonstrates the error.

**What this teaches**
- Confirms that window functions are not allowed in HAVING.
- The error is raised at parse/analysis time, before any data is touched.


## Exercise 2 – Pattern A: HAVING first, then window

**Business question:**  
Among countries that have **more than 3** customers, rank those countries by customer count.

### Instructions

1. Group by `Country`
2. Keep only groups with `COUNT(*) > 3` (HAVING)
3. In the SELECT list, also compute `RANK() OVER (ORDER BY COUNT(*) DESC)`
4. Order the final result by the rank


In [ ]:
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM customers
GROUP BY Country
HAVING COUNT(*) > 3
ORDER BY Rank;


**Solution**

```sql
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM customers
GROUP BY Country
HAVING COUNT(*) > 3
ORDER BY Rank;
```

**Hints**
```sql
SELECT
    Country,
    COUNT(*) AS CustomerCount,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM customers
GROUP BY Country
HAVING COUNT(*) > 3
ORDER BY Rank;
```
The window runs only on the groups that survived HAVING.


## Exercise 3 – Pattern B: Filter on a window result (CTE)

**Business question:**  
Show only the **top 3** countries by number of customers (using a window rank).

Because we cannot put the rank in HAVING, we wrap the query in a CTE (or subquery) and filter afterwards.

### Instructions

Write a query that:

1. In a CTE, groups customers by country, computes the count and a `RANK()`
2. In the outer query, keeps only rows where rank ≤ 3
3. Orders by rank


In [ ]:
WITH ranked AS (
    SELECT
        Country,
        COUNT(*) AS CustomerCount,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
    FROM customers
    GROUP BY Country
)
SELECT *
FROM ranked
WHERE Rank <= 3
ORDER BY Rank;


**Solution**

```sql
WITH ranked AS (
    SELECT
        Country,
        COUNT(*) AS CustomerCount,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
    FROM customers
    GROUP BY Country
)
SELECT *
FROM ranked
WHERE Rank <= 3
ORDER BY Rank;
```

**Hints**
```sql
WITH ranked AS (
    SELECT
        Country,
        COUNT(*) AS CustomerCount,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
    FROM customers
    GROUP BY Country
)
SELECT *
FROM ranked
WHERE Rank <= 3
ORDER BY Rank;
```
This is the standard SQLite substitute for the `QUALIFY` clause that some other databases offer.


## Exercise 4 – Top-N per group with windows (no HAVING needed)

**Business question:**  
For each billing country, show the **two highest** invoice totals.

### Instructions

Use `ROW_NUMBER()` partitioned by country:

1. Create a CTE that selects from `invoices` and adds  
   `ROW_NUMBER() OVER (PARTITION BY BillingCountry ORDER BY Total DESC) AS rn`
2. Outer query keeps only `rn <= 2`
3. Order by country and total descending


In [ ]:
WITH ranked_invoices AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT BillingCountry, InvoiceId, Total
FROM ranked_invoices
WHERE rn <= 2
ORDER BY BillingCountry, Total DESC;


**Solution**

```sql
WITH ranked_invoices AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT BillingCountry, InvoiceId, Total
FROM ranked_invoices
WHERE rn <= 2
ORDER BY BillingCountry, Total DESC;
```

**Hints**
```sql
WITH ranked_invoices AS (
    SELECT
        BillingCountry,
        InvoiceId,
        Total,
        ROW_NUMBER() OVER (
            PARTITION BY BillingCountry
            ORDER BY Total DESC
        ) AS rn
    FROM invoices
)
SELECT BillingCountry, InvoiceId, Total
FROM ranked_invoices
WHERE rn <= 2
ORDER BY BillingCountry, Total DESC;
```
Here HAVING is not involved at all — the window + outer filter replaces a classic “top-N per group” problem.


## Exercise 5 – Combine GROUP BY + HAVING + Window

**Business question:**  
For support representatives who generated **more than $700** in revenue,  
show their total revenue and their rank among those high-performing reps.

### Instructions

1. Join `customers` and `invoices`
2. Group by `SupportRepId`
3. HAVING `SUM(Total) > 700`
4. Add a `RANK() OVER (ORDER BY SUM(Total) DESC)` in the SELECT
5. Order by rank


In [ ]:
SELECT
    c.SupportRepId,
    SUM(i.Total) AS Revenue,
    RANK() OVER (ORDER BY SUM(i.Total) DESC) AS RevenueRank
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.SupportRepId
HAVING SUM(i.Total) > 700
ORDER BY RevenueRank;


**Solution**

```sql
SELECT
    c.SupportRepId,
    SUM(i.Total) AS Revenue,
    RANK() OVER (ORDER BY SUM(i.Total) DESC) AS RevenueRank
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.SupportRepId
HAVING SUM(i.Total) > 700
ORDER BY RevenueRank;
```

**Hints**
```sql
SELECT
    c.SupportRepId,
    SUM(i.Total) AS Revenue,
    RANK() OVER (ORDER BY SUM(i.Total) DESC) AS RevenueRank
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.SupportRepId
HAVING SUM(i.Total) > 700
ORDER BY RevenueRank;
```


## Exercise 6 – Running total after filtering groups

**Business question:**  
List billing countries with **at least 20** invoices.  
For those countries, also show a running total of revenue ordered by country name.

### Instructions

1. Group by `BillingCountry`
2. HAVING `COUNT(*) >= 20`
3. In SELECT compute:
   - `SUM(Total)` as Revenue
   - `SUM(SUM(Total)) OVER (ORDER BY BillingCountry) AS RunningRevenue`
4. Order by country


In [ ]:
SELECT
    BillingCountry,
    SUM(Total) AS Revenue,
    SUM(SUM(Total)) OVER (ORDER BY BillingCountry) AS RunningRevenue
FROM invoices
GROUP BY BillingCountry
HAVING COUNT(*) >= 20
ORDER BY BillingCountry;


**Solution**

```sql
SELECT
    BillingCountry,
    SUM(Total) AS Revenue,
    SUM(SUM(Total)) OVER (ORDER BY BillingCountry) AS RunningRevenue
FROM invoices
GROUP BY BillingCountry
HAVING COUNT(*) >= 20
ORDER BY BillingCountry;
```

**Hints**
- Nested aggregate `SUM(SUM(Total)) OVER (...)` is valid: the inner SUM is the group aggregate, the outer SUM is the window.
- The window only sees the groups that passed HAVING.


## Exercise 7 – Compare rank before vs after HAVING

**Goal:** Understand that the window ranking changes depending on whether you rank *all* groups or only the filtered ones.

### Instructions

Write **two** queries side-by-side (two cells):

**Query A** – Rank *all* countries, then keep rank ≤ 5 in an outer query.  
**Query B** – First HAVING `COUNT(*) > 2`, then rank the remaining countries.

Observe that the ranks (and which countries appear) differ.


In [ ]:
-- Query A: rank ALL countries, then keep top 5
WITH all_ranked AS (
    SELECT
        Country,
        COUNT(*) AS cnt,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
    FROM customers
    GROUP BY Country
)
SELECT *
FROM all_ranked
WHERE rnk <= 5
ORDER BY rnk;


In [ ]:
-- Query B: filter first (HAVING), then rank the survivors
SELECT
    Country,
    COUNT(*) AS cnt,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
FROM customers
GROUP BY Country
HAVING COUNT(*) > 2
ORDER BY rnk;


**Solutions**

**Query A:**
```sql
-- Query A: rank ALL countries, then keep top 5
WITH all_ranked AS (
    SELECT
        Country,
        COUNT(*) AS cnt,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
    FROM customers
    GROUP BY Country
)
SELECT *
FROM all_ranked
WHERE rnk <= 5
ORDER BY rnk;
```

**Query B:**
```sql
-- Query B: filter first (HAVING), then rank the survivors
SELECT
    Country,
    COUNT(*) AS cnt,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
FROM customers
GROUP BY Country
HAVING COUNT(*) > 2
ORDER BY rnk;
```

**Hints – Query A**
```sql
WITH all_ranked AS (
    SELECT Country, COUNT(*) AS cnt,
           RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
    FROM customers
    GROUP BY Country
)
SELECT * FROM all_ranked WHERE rnk <= 5 ORDER BY rnk;
```

**Hints – Query B**
```sql
SELECT Country, COUNT(*) AS cnt,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
FROM customers
GROUP BY Country
HAVING COUNT(*) > 2
ORDER BY rnk;
```


## Exercise 8 – Window on a filtered set of invoices

**Business question:**  
Considering only invoices with `Total > 10`,  
for each billing country compute:
- the number of such high-value invoices
- the rank of the country by that count

Keep only countries that have **at least 2** high-value invoices.

### Instructions

1. `WHERE Total > 10` (row filter)
2. `GROUP BY BillingCountry`
3. `HAVING COUNT(*) >= 2`
4. Add `RANK() OVER (ORDER BY COUNT(*) DESC)`


In [ ]:
SELECT
    BillingCountry,
    COUNT(*) AS HighValueInvoices,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM invoices
WHERE Total > 10
GROUP BY BillingCountry
HAVING COUNT(*) >= 2
ORDER BY Rank;


**Solution**

```sql
SELECT
    BillingCountry,
    COUNT(*) AS HighValueInvoices,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM invoices
WHERE Total > 10
GROUP BY BillingCountry
HAVING COUNT(*) >= 2
ORDER BY Rank;
```

**Hints**
```sql
SELECT
    BillingCountry,
    COUNT(*) AS HighValueInvoices,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS Rank
FROM invoices
WHERE Total > 10
GROUP BY BillingCountry
HAVING COUNT(*) >= 2
ORDER BY Rank;
```
Classic three-layer filter: WHERE → HAVING → (window for ranking).


## Exercise 9 – Dense rank of genres by average track length

**Business question:**  
Rank genres by their average track duration (longest first).  
Show only genres whose average duration is **greater than 300 000 ms** (~5 minutes).

### Instructions

Join `tracks` and `genres`, group by genre name,  
use HAVING on the average, and add `DENSE_RANK()`.


In [ ]:
SELECT
    g.Name AS Genre,
    AVG(t.Milliseconds) AS AvgDuration,
    DENSE_RANK() OVER (ORDER BY AVG(t.Milliseconds) DESC) AS DurationRank
FROM tracks t
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING AVG(t.Milliseconds) > 300000
ORDER BY DurationRank;


**Solution**

```sql
SELECT
    g.Name AS Genre,
    AVG(t.Milliseconds) AS AvgDuration,
    DENSE_RANK() OVER (ORDER BY AVG(t.Milliseconds) DESC) AS DurationRank
FROM tracks t
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING AVG(t.Milliseconds) > 300000
ORDER BY DurationRank;
```

**Hints**
```sql
SELECT
    g.Name AS Genre,
    AVG(t.Milliseconds) AS AvgDuration,
    DENSE_RANK() OVER (ORDER BY AVG(t.Milliseconds) DESC) AS DurationRank
FROM tracks t
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
HAVING AVG(t.Milliseconds) > 300000
ORDER BY DurationRank;
```


## Exercise 10 – Challenge: Top revenue country per year

**Business question:**  
For each year, which billing country generated the highest revenue?  
Return only that top country per year.

### Instructions

1. Create a CTE that groups by year + country, computing revenue,  
   and assigns `ROW_NUMBER() OVER (PARTITION BY year ORDER BY revenue DESC)`
2. Outer query keeps `rn = 1`
3. Order by year

(You may also experiment with adding a HAVING inside the CTE if you want to exclude very small countries.)


In [ ]:
WITH yearly AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
)
SELECT Year, BillingCountry, Revenue
FROM yearly
WHERE rn = 1
ORDER BY Year;


**Solution**

```sql
WITH yearly AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
)
SELECT Year, BillingCountry, Revenue
FROM yearly
WHERE rn = 1
ORDER BY Year;
```

**Hints**
```sql
WITH yearly AS (
    SELECT
        strftime('%Y', InvoiceDate) AS Year,
        BillingCountry,
        SUM(Total) AS Revenue,
        ROW_NUMBER() OVER (
            PARTITION BY strftime('%Y', InvoiceDate)
            ORDER BY SUM(Total) DESC
        ) AS rn
    FROM invoices
    GROUP BY strftime('%Y', InvoiceDate), BillingCountry
)
SELECT Year, BillingCountry, Revenue
FROM yearly
WHERE rn = 1
ORDER BY Year;
```


## Summary – Decision Guide

| Goal | Technique |
|------|-----------|
| Filter groups by aggregate | `HAVING` |
| Rank / number the groups that survived HAVING | Window in the same SELECT |
| Filter *on* a window result (top-N, rank ≤ k) | CTE / subquery + outer `WHERE` |
| Top-N **per partition** | `ROW_NUMBER() OVER (PARTITION BY …)` + outer filter |
| Running totals after group filter | `SUM(SUM(…)) OVER (ORDER BY …)` after HAVING |

### Execution order (remember this)

```
FROM → WHERE → GROUP BY → HAVING → WINDOW → SELECT → ORDER BY → LIMIT
```

Window functions are calculated **after** HAVING, which is why they cannot appear inside HAVING itself.

### SQLite vs other engines

- SQLite, PostgreSQL, SQL Server, MySQL 8+: window functions **not** allowed in HAVING/WHERE.
- Snowflake, BigQuery, DuckDB and some others offer a `QUALIFY` clause that filters on window results directly — the CTE pattern above is the portable equivalent.
